# Data

In [ ]:
import os
import numpy as np 
import pandas as pd 
pd.pandas.set_option('display.max_columns',None)
pd.pandas.set_option('display.max_rows',None)

In [ ]:
train_path = "/kaggle/input/playground-series-s6e2/train.csv"
test_path = "/kaggle/input/playground-series-s6e2/test.csv"
submission_path = "/kaggle/input/playground-series-s6e2/sample_submission.csv"

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
submission = pd.read_csv(submission_path)

train.head(2)

In [ ]:
print("The train data shape is {}".format(train.shape))
print("The test data shape is {}".format(test.shape))

In [ ]:
remove_features=['id']
train=train.drop(columns=remove_features,axis=1)
test=test.drop(columns=remove_features,axis=1)

# Target Distribution

In [ ]:
train["Heart Disease"].value_counts()

In [ ]:
# presence=yes

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

# Counting the observations for each category
LABEL="Heart Disease"
status_counts = train[LABEL].value_counts()
labels = status_counts.index
sizes = status_counts.values

# Calculating the percentage of each category
percentages = 100.*sizes/sizes.sum()

# Creating the pie chart with percentages in the labels
plt.figure(figsize=(10, 6))
plt.pie(sizes, labels=[f"{l}, {s:.1f}%" for l, s in zip(labels, percentages)], startangle=90)
plt.gca().set_aspect("equal")
plt.legend(loc="upper right", bbox_to_anchor=(1.2, 1), labels=labels, title=LABEL)
plt.title(f"Distribution of {LABEL}")
plt.show()

In [ ]:
target_label={
    "Presence":1,
    "Absence":0
}

train["Heart Disease"] = train["Heart Disease"].map(target_label).astype(int)

# Unique & Missing Values

In [ ]:
from prettytable import PrettyTable

target="Heart Disease"
train_copy = train.copy()
test_copy = test.copy()

# create a table
table = PrettyTable()

# columns names
table.field_names = ['Feature', 'Data Type', 'Train Missing %', 'Test Missing %']

for column in train_copy.columns:
    data_type = str(train_copy[column].dtype)
    non_null_count_train= np.round(100-train_copy[column].count()/train_copy.shape[0]*100,1)
    if column!=target:
        non_null_count_test = np.round(100-test_copy[column].count()/test_copy.shape[0]*100,1)
    else:
        non_null_count_test="NA"
    #non_null_count_orig= np.round(100-original_copy[column].count()/original_copy.shape[0]*100,1)
    table.add_row([column, data_type, non_null_count_train,non_null_count_test])
print(table)

In [ ]:
def describe_features(df,features):
    if isinstance(features,str):
        featrues = [features]

    summary = []
    for col in features:
        if col in df.columns:
            summary.append({
                "Feature":col,
                "Type":df[col].dtype,
                "Unique Values":df[col].nunique(),
                #"Missing Value":df[col].isna().sum()
            })
        else:
            summary.append({
                "Feature":col,
                "Type":str(df[col].dtype),
                "Unique Values":None,
                #"Missing Value":None
            })
    return pd.DataFrame(summary)

train_features = train.columns.tolist()
describe_features(train, train_features)

# Features Distribution

In [ ]:
cate_cols = [
    col for col in train.columns
    if (train[col].dtype == "object" or train[col].nunique()<10)
    and col != "Heart Disease"
]

cate_cols

In [ ]:
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

LBEL = "Heart Disease"

plt.figure(figsize=(14, len(cate_cols) * 2))
for i, col in enumerate(cate_cols):
    plt.subplot(len(cate_cols) // 2 + 1, 3, i + 1)
    sns.countplot(x=col, hue=LABEL, data=train)
    plt.title(f"{col} vs {LABEL}")
    plt.tight_layout()

In [ ]:
nums_cols = ["Age","BP","Cholesterol","Max HR","ST depression"]

def plot_kde_by_label(df, nums_cols, label_col, n_cols=3, figsize_per_plot=(5,3), alpha=0.3):
    n_rows = int(len(nums_cols) / n_cols) + (len(nums_cols) % n_cols != 0)
    plt.figure(figsize=(figsize_per_plot[0]*n_cols, figsize_per_plot[1]*n_rows))

    for i, col in enumerate(nums_cols, 1):
        plt.subplot(n_rows, n_cols, i)
        for label in df[label_col].unique():
            subset = df[df[label_col] == label]
            sns.kdeplot(subset[col], label=f"{label_col}={label}", fill=True, alpha=alpha)
        
        plt.title(f"{col} - KDE Distribution by {label_col}")
        plt.xlabel(col)
        plt.ylabel("Density")
        plt.legend()
    
    plt.tight_layout()
    plt.show()
    
plot_kde_by_label(train, nums_cols, LABEL)

In [ ]:
#pair_plot_cols=nums_cols

#sns.set(font_scale=1)
#plt.figure(figsize=(18, 10))
#sns.set(style="ticks", color_codes=True)
#sns.pairplot(data=train, vars=pair_plot_cols,diag_kind='kde', 
        #kind='scatter', palette='muted', 
        #plot_kws={'s': 20}, hue='Heart Disease')
#plt.show()

# Error-Driven Features

# Numerical — Rolling + Decision Stump

In [ ]:
from sklearn.tree import DecisionTreeClassifier
import math

def plot_rolling_with_stump_multi(
    df,
    features,
    target,
    window=500,
    min_periods=100,
    min_samples_leaf=200,
    n_cols=3
):
    """
    Rolling target mean + decision stump visualization (multi-feature)

    Returns
    -------
    results : list of dict
        Raw per-feature split statistics
    summary_df : pd.DataFrame
        Sorted, labeled summary table for decision making
    """

    n_features = len(features)
    n_rows = math.ceil(n_features / n_cols)

    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(n_cols * 5, n_rows * 4),
        squeeze=False
    )

    results = []

    for idx, feature in enumerate(features):
        r, c = divmod(idx, n_cols)
        ax = axes[r][c]

        tmp = df[[feature, target]].dropna().sort_values(feature)

        if tmp.empty or tmp[feature].nunique() < 2:
            ax.set_title(f'{feature} (no valid data)')
            ax.axis('off')
            continue

        # ---------- rolling target mean ----------
        rolling_mean = (
            tmp[target]
            .rolling(
                window=window,
                min_periods=min_periods,
                center=True
            )
            .mean()
        )

        # ---------- decision stump ----------
        tree = DecisionTreeClassifier(
            max_depth=1,
            min_samples_leaf=min_samples_leaf,
            random_state=42
        )
        tree.fit(tmp[[feature]], tmp[target])

        threshold = tree.tree_.threshold[0]

        # ---------- edge split detection ----------
        f_min = tmp[feature].min()
        f_max = tmp[feature].max()
        span = f_max - f_min + 1e-9

        edge_ratio = min(
            (threshold - f_min) / span,
            (f_max - threshold) / span
        )
        edge_split = edge_ratio < 0.05

        # ---------- plot ----------
        ax.plot(tmp[feature], rolling_mean, label='Rolling mean')
        ax.axvline(
            threshold,
            color='red',
            linestyle='--',
            label=f'split @ {threshold:.2f}'
        )

        ax.set_title(feature)
        ax.set_xlabel(feature)
        ax.set_ylabel('Event rate')
        ax.grid(alpha=0.3)

        # ---------- stats ----------
        left = tmp[tmp[feature] <= threshold][target]
        right = tmp[tmp[feature] > threshold][target]

        results.append({
            'feature': feature,
            'threshold': threshold,
            'left_n': len(left),
            'left_event_rate': left.mean(),
            'right_n': len(right),
            'right_event_rate': right.mean(),
            'diff': right.mean() - left.mean(),
            'abs_diff': abs(right.mean() - left.mean()),
            'min_leaf_n': min(len(left), len(right)),
            'edge_split': edge_split
        })

    # ---------- remove empty subplots ----------
    for j in range(idx + 1, n_rows * n_cols):
        r, c = divmod(j, n_cols)
        fig.delaxes(axes[r][c])

    plt.tight_layout()
    plt.show()

    # ================= summary table =================
    summary_df = pd.DataFrame(results)

    def signal_label(row):
        if row['min_leaf_n'] < min_samples_leaf:
            return 'too_small'
        if row['edge_split']:
            return 'edge'
        if row['abs_diff'] > 0.02:
            return 'strong'
        if row['abs_diff'] > 0.005:
            return 'weak'
        return 'none'

    summary_df['signal'] = summary_df.apply(signal_label, axis=1)

    # ---------- correct sorting ----------
    signal_order = {
        'strong': 0,
        'weak': 1,
        'none': 2,
        'edge': 3,
        'too_small': 4
    }

    summary_df['signal_rank'] = summary_df['signal'].map(signal_order)

    summary_df = (
        summary_df
        .sort_values(
            ['signal_rank', 'abs_diff'],
            ascending=[True, False]
        )
        .reset_index(drop=True)
        .drop(columns='signal_rank')
    )

    return results, summary_df


results, summary_df = plot_rolling_with_stump_multi(train,features=nums_cols,target='Heart Disease',window=500,min_samples_leaf=200,n_cols=3)
summary_df

# Numerical — Quantile Stability Curve

In [ ]:
def plot_continuous_event_rate_multi_row(df, features, target, n_bins=10, show_count=True, n_cols=3):
    
    n_features = len(features)
    n_rows = (n_features + n_cols - 1) // n_cols  # 計算行數
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols*5, n_rows*4))
    axes = axes.flatten()
    
    for i, feature in enumerate(features):
        # 分箱
        df['bin'] = pd.qcut(df[feature], q=n_bins, duplicates='drop')
        stats = df.groupby('bin')[target].agg(['mean','count']).reset_index()
        
        ax1 = axes[i]
        ax1.plot(stats['bin'].astype(str), stats['mean'], marker='o', color='blue')
        ax1.set_xlabel(feature)
        ax1.set_ylabel('Event rate', color='blue')
        ax1.tick_params(axis='x',rotation=45)
        ax1.set_title(f'{feature} vs {target}')
        
        if show_count:
            ax2 = ax1.twinx()
            ax2.bar(stats['bin'].astype(str), stats['count'], alpha=0.3, color='grey')
            ax2.set_ylabel('Count', color='grey')
    
    
    for j in range(i+1, len(axes)):
        fig.delaxes(axes[j])
    
    plt.tight_layout()
    plt.show()
    df.drop(columns=['bin'], inplace=True)

plot_continuous_event_rate_multi_row(train, nums_cols, target='Heart Disease', n_bins=10, n_cols=3)

# Numerical — Local ROC-AUC Contribution

In [ ]:
from sklearn.model_selection import StratifiedKFold
from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score

def numerical_local_auc_contribution(
    X: pd.DataFrame,
    y: pd.Series,
    numerical_features: list,
    model,
    n_splits: int = 5,
    n_bins: int = 10,
    random_state: int = 42,
    plot: bool = True,
    y_col: str = "y"
):

    # ---------- Step 1: OOF prediction ----------
    oof_pred = np.zeros(len(X))
    skf = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_state
    )

    for tr_idx, val_idx in skf.split(X, y):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr = y.iloc[tr_idx]

        model.fit(X_tr, y_tr)
        oof_pred[val_idx] = model.predict_proba(X_val)[:, 1]

    global_auc = roc_auc_score(y, oof_pred)

    base_df = pd.DataFrame({
        y_col: y.values,
        "pred": oof_pred
    })

    results = {}

    # ---------- Step 2: Per-feature local AUC ----------
    for feat in numerical_features:
        df = base_df.copy()
        df["feature"] = X[feat].values

        df["bin"] = pd.qcut(df["feature"], q=n_bins, duplicates="drop")

        rows = []
        for b, g in df.groupby("bin"):
            if g[y_col].nunique() < 2:
                local_auc = np.nan
            else:
                local_auc = roc_auc_score(g[y_col], g["pred"])
            rows.append({
                "bin": b,
                "bin_mid": (b.left + b.right) / 2,
                "local_auc": local_auc,
                "auc_drop": global_auc - local_auc if not np.isnan(local_auc) else np.nan,
                "sample_count": len(g)
            })

        results[feat] = pd.DataFrame(rows)

    # ---------- Step 3: Plot all features ----------
    if plot:
        n_feats = len(numerical_features)
        n_cols = 3
        n_rows = (n_feats + n_cols - 1) // n_cols

        fig, axes = plt.subplots(n_rows, n_cols, figsize=(6*n_cols, 4*n_rows))
        axes = axes.flatten()

        for i, feat in enumerate(numerical_features):
            out = results[feat]
            ax = axes[i]

            ax.plot(out["bin_mid"], out["local_auc"], marker="o", label="Local ROC-AUC")
            ax.axhline(global_auc, linestyle="--", color="gray", alpha=0.6,
                       label=f"Global AUC = {global_auc:.4f}")
            ax.axhline(0.5, linestyle=":", color="red", alpha=0.5)
            ax.set_title(feat)
            ax.set_xlabel("Feature value (quantile bins)")
            ax.set_ylabel("ROC-AUC")
            ax.grid(alpha=0.3)
            ax.legend()

        
        for j in range(i+1, len(axes)):
            axes[j].axis("off")

        plt.tight_layout()
        plt.show()

    return results

results = numerical_local_auc_contribution(
    X=train[nums_cols],
    y=train["Heart Disease"],
    numerical_features=nums_cols,
    model=LGBMClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        random_state=42,
        verbose=-1
    ),
    n_bins=10,
    plot=True,
    y_col='y'
)

In [ ]:
#Max HR / ST depression
# section features or log/square transformer
# interaction with other features
# bins 

# Categorical — CV-safe Rank Consistency Probe

In [ ]:
from sklearn.model_selection import StratifiedKFold
from scipy.stats import rankdata, spearmanr

def cv_safe_rank_consistency_single(
    X: pd.DataFrame,
    y: pd.Series,
    cat_col: str,
    n_splits: int = 5,
    random_state: int = 42
):
    """
    CV-safe Categorical Rank Consistency Probe (single feature)
    """
    skf = StratifiedKFold(n_splits=n_splits, random_state=random_state, shuffle=True)
    oof_rank = np.zeros(len(X))

    global_mean = y.mean()

    for train_idx, val_idx in skf.split(X,y):
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr = y.iloc[train_idx]

        # category -> target mean (train only)
        cat_mean = (
            pd.concat([X_tr[[cat_col]], y_tr], axis=1)
            .groupby(cat_col)[y.name]
            .mean()
        )

        # map to validation (unseen -> global mean)
        mapped = X_val[cat_col].map(cat_mean).fillna(global_mean)

        # rank inside validation fold
        oof_rank[val_idx] = rankdata(mapped)

    # rank y globally
    y_rank = rankdata(y)

    spearman_corr = spearmanr(oof_rank, y_rank).correlation
    return spearman_corr

def cv_safe_rank_consistency_multi(
    X: pd.DataFrame,
    y: pd.Series,
    cat_cols: list,
    n_splits: int = 5,
    random_state: int = 42
):
    """
    CV-safe Categorical Rank Consistency Probe (multiple features)
    """
    results = {}

    for col in cat_cols:
        score = cv_safe_rank_consistency_single(
            X=X,
            y=y,
            cat_col=col,
            n_splits=n_splits,
            random_state=random_state
        )
        results[col] = score

    return (
        pd.DataFrame
        .from_dict(results, orient="index", columns=["cv_rank_consistency"])
        .sort_values("cv_rank_consistency", ascending=False)
    )


rank_probe_df = cv_safe_rank_consistency_multi(
    X=train,
    y=train["Heart Disease"],
    cat_cols=cate_cols,
    n_splits=5
)

print(rank_probe_df)

# Outiler Remove

In [ ]:
print("Before remove outlier train data shape is {}".format(train.shape))

tmp_df = train.copy()

means = tmp_df[nums_cols].mean()
std_devs = tmp_df[nums_cols].std()

n_stds = 6
thresholds = n_stds * std_devs

outliers = (np.abs(tmp_df[nums_cols] - means) > thresholds).any(axis=1)

print(f"Detected {sum(outliers)} that are more than {n_stds} SDs away from mean...")

In [ ]:
outliers_df = tmp_df[outliers]

train = tmp_df[~outliers].reset_index(drop=True)

print("After remove outlier train data shape is {}".format(train.shape))

# New Features

In [ ]:
def funtional_features(df):
    #
    log_cols=['ST depression','Cholesterol','BP']
    for col in log_cols:
        df[f"{col}_log1p"] = np.log1p(df[col])
    #
    poly_cols=['Age', 'BP', 'Cholesterol', 'Max HR']
    for col in poly_cols:
        df[f"{col}_pow2"] = df[col] ** 2

    df['log_cholerterol_BP'] = df["Cholesterol_log1p"] + df["BP_log1p"]
    df["HR_ST"] = (df["ST depression"] + 1e-5) / df["Max HR"]
    df["Age_Max_HR"] = 220 - (df["Age"] - df["Max HR"])
    df["is_ST_depressed"] = (df["ST depression"] > 0.5).astype(int)
    df["Age_in_45-60"] = ((df["Age"] >=45) & (df["Age"] <= 65)).astype(int)
    df["Risk_score"] = ((df["BP"] > 140) & (df["Age"] > 50) & (df["Cholesterol"] > 240)).astype(int)
    df["BP_per_Age"] = df["BP"] / df["Age"]
    df["BP_mul_Age"] = df["BP"] * df["Age"]
    df["BP_mul_MaxHR"] = df["BP"] * df["Max HR"]
    df["BP_per_MaxHR"] = df["BP"] / df["Max HR"]
    df["Cholesterol_mul_BP"] = df["Cholesterol"] * df["BP"]
    df["lower_O2"] = ((df["ST depression"] > 0) & (df["Max HR"] < 140)).astype(int)
    df["CVI"] = (df["Age"] * df["BP"]) / df["Max HR"]
    df["High_risk_flag"] = ((df["Age"] >= 55) & (df["Cholesterol"] >=263) & (df["ST depression"] > 1)).astype(int)
    df["Strain"] = (df["ST depression"] + 1e-5) / df["Max HR"]
    return df

train = funtional_features(train)
test = funtional_features(test)

In [ ]:
def add_scaled_row_stats(train_df, test_df, features):

    train_df = train_df.copy()
    test_df = test_df.copy()

    
    for col in features:
        t_min = train_df[col].min()
        t_max = train_df[col].max()
        t_range = t_max - t_min
        
        
        if t_range == 0:
            train_df[f"{col}_norm"] = 0
            test_df[f"{col}_norm"] = 0
        else:
            train_df[f"{col}_norm"] = (train_df[col] - t_min) / t_range
            test_df[f"{col}_norm"] = (test_df[col] - t_min) / t_range

    
    norm_features = [f"{col}_norm" for col in features]
    
    def get_row_stats(df):
        subset = df[norm_features]
        df["row_mean"] = subset.mean(axis=1)
        df["row_std"]  = subset.std(axis=1)
        df["row_min"]  = subset.min(axis=1)
        df["row_max"]  = subset.max(axis=1)
        df["row_range"] = df["row_max"] - df["row_min"]
        df["row_cv"]   = df["row_std"] / (df["row_mean"] + 1e-6)
        return df

    train_df = get_row_stats(train_df)
    test_df = get_row_stats(test_df)

    return train_df, test_df

row_stats_features = ['Age', 'BP', 'Cholesterol', 'Max HR', 'ST depression','Slope of ST','Number of vessels fluro']
train,test = add_scaled_row_stats(train,test,features=row_stats_features)

# Numerical Features Transform

In [ ]:
from scipy.stats import skew
import math

def analyze_numeric_features(df, numeric_cols, plot=True):
    results = []
    n_cols = len(numeric_cols)
    
    cols_per_row = 3
    n_rows = math.ceil(n_cols / cols_per_row)

    if plot:
        fig, axes = plt.subplots(n_rows, cols_per_row, figsize=(18, 5 * n_rows))
        axes = axes.flatten()  #

    for i, col in enumerate(numeric_cols):
        data = df[col].dropna()
        skewness = skew(data)

        if abs(skewness) < 0.5:
            suggestion = "StandardScaler"
        elif abs(skewness) < 1.0:
            suggestion = "MinMaxScaler"
        elif abs(skewness) < 2.0:
            suggestion = "RobustScaler"
        else:
            suggestion = "Box-Cox / Yeo-Johnson"

        results.append({
            "feature": col,
            "skewness": skewness,
            "suggestion": suggestion
        })

        if plot:
            ax = axes[i]
            sns.histplot(data, kde=True, bins=50, color='skyblue', ax=ax)
            ax.set_title(f"{col}\nSkew: {skewness:.2f}\nUse: {suggestion}", fontsize=12)
            ax.set_xlabel("")
            ax.set_ylabel("Count")

    if plot:
        for j in range(i + 1, len(axes)):
            axes[j].axis('off')
        
        plt.tight_layout()
        plt.show()
    
    return pd.DataFrame(results)


result_df = analyze_numeric_features(train, nums_cols)
print(result_df)

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.preprocessing import PowerTransformer

def scale_selected_features(train_df, test_df, feature_suggestions):
    
     # copy
    scaled_train = train_df.copy()
    scaled_test = test_df.copy()
    
    for suggestion, group in feature_suggestions.groupby('suggestion'):
        features = group['feature'].tolist()
        
        if suggestion == "StandardScaler (Z-score)":
            scaler = StandardScaler()
            suffix = "_std"
        elif suggestion == "MinMaxScaler":
            scaler = MinMaxScaler()
            suffix = "_minmax"
        elif suggestion == "RobustScaler":
            scaler = RobustScaler()
            suffix = "_robust"
        elif suggestion == "Box-Cox / Yeo-Johnson":
            scaler = PowerTransformer(method='yeo-johnson')  # Box-Cox >0
            suffix = "_power"
        else:
            scaler = RobustScaler()
            suffix = "_scaled"

        new_features_names = [f"{feat}{suffix}" for feat in features]
        # fit on train
        scaler.fit(train_df[features])
        # transform train/test
        scaled_train[new_features_names] = scaler.transform(train_df[features])
        scaled_test[new_features_names] = scaler.transform(test_df[features])
    
    return scaled_train, scaled_test

train, test = scale_selected_features(train, test, result_df)

# Create Features from Rolling + Decision Stump

In [ ]:
def decision_features(df, train_stats=None):
    new_df = df.copy()
    
    # thresholds
    thresholds = {
        "ST depression": 0.95, "Max HR": 147.5, "Age": 53.5, 
        "Cholesterol": 241.0, "BP": 161.0
    }
    for feat, thresh in thresholds.items():
        new_df[f"{feat}_threshold"] = (new_df[feat] >= thresh).astype(int)
        # how far with thresholds
        new_df[f"{feat}_dist_thresh"] = new_df[feat] - thresh

    # signal features weight
    diff_weights = {
        "ST depression": 0.443772, "Max HR": 0.414277, "Age": 0.213022, 
        "Cholesterol": 0.088521, "BP": 0.067094
    }
    for feat, weight in diff_weights.items():
        new_df[f"{feat}_abs_diff"] = new_df[feat] * weight

    # how far with mean
    for feat in thresholds.keys():
        if train_stats and feat in train_stats:
            m = train_stats[feat]['mean']
            s = train_stats[feat]['std']
        else:
            m = new_df[feat].mean()
            s = new_df[feat].std()
            
        new_df[f"{feat}_far_mean"] = (new_df[feat] - m) / (s + 1e-5)
        
    return new_df

train = decision_features(train)
test = decision_features(test)

# Create Features from Local ROC-AUC Contribution 

In [ ]:
def create_custom_bins(train,test,col_configs):
    for col,bins in col_configs.items():
        actual_bins = sorted(list(set([-np.inf] + bins + [np.inf])))

        labels = range(1,len(actual_bins))
        train[f"{col}_custom_bin"] = pd.cut(train[col],bins=actual_bins,labels=labels).astype(int)
        test[f"{col}_custom_bin"] = pd.cut(test[col],bins=actual_bins,labels=labels).astype(int)

    return train,test

custom_configs = {
    "Max HR":[100,130,140,150,160,170,185],
    "ST depression":[0.5,1.0,1.5,2.0,4.0],
    "Age":[45,55,65],
    "BP":[120,140,160],
    "Cholesterol":[200,240,280,320]
}

train,test = create_custom_bins(train,test,custom_configs)

In [ ]:
from itertools import combinations
nums_bin_features = ["Max HR_custom_bin","ST depression_custom_bin","Age_custom_bin","BP_custom_bin","Cholesterol_custom_bin"]
comb_features = cate_cols + nums_bin_features

def pair_features(train_df,test_df,pair_features,sep="_"):
    new_features=[]

    for f1,f2 in combinations(pair_features,2):
        col_name = f"{f1}_{f2}"
        print("Create {}".format(col_name))
        train[col_name] = train[f1].astype(str) + sep + train[f2].astype(str)
        test[col_name] = test[f1].astype(str) + sep + test[f2].astype(str)
        new_features.append(col_name)

    return train,test,new_features

train,test,features_combine = pair_features(train_df=train,test_df=test,pair_features=comb_features)
print("Create {} Features".format(len(features_combine)))

# Different Encode

In [ ]:
custom_bins = ["Max HR_custom_bin","ST depression_custom_bin","Age_custom_bin","BP_custom_bin","Cholesterol_custom_bin"]
frequency_cols= cate_cols + custom_bins

def frequency_encode(train_df, test_df, cat_cols, suffix="_freq", normalize=True):
    train_df = train_df.copy()
    test_df = test_df.copy()

    n = len(train_df)

    for col in cat_cols:
        new_col = col + suffix

        freq = train_df[col].value_counts()
        if normalize:
            freq = freq / n

        train_df[new_col] = train_df[col].map(freq)
        test_df[new_col] = test_df[col].map(freq).fillna(0)

    return train_df, test_df

train,test = frequency_encode(train_df=train,test_df=test,cat_cols=frequency_cols)

In [ ]:
te_cols = train.select_dtypes(include='object').columns.tolist()
te_cols = te_cols + cate_cols
print("Have {} features to target encode".format(len(te_cols)))

def target_encode(train_df, test_df, cat_cols, target_col="Heart Disease", n_splits=5, smooth=20, random_state=42):
    
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    global_mean = train_df[target_col].mean()

    # Target Encoding
    for feature in cat_cols:
        print(f"Encoding: {feature}")
        train_encoded = pd.Series(index=train_df.index, dtype=float)
        test_encoded = pd.Series(index=test_df.index, dtype=float)

        # train TE (OOF)
        for train_idx, val_idx in skf.split(train_df,train_df[target_col]):
            tr, val = train_df.iloc[train_idx], train_df.iloc[val_idx]
            stats = tr.groupby(feature)[target_col].agg(['mean', 'count'])
            smooth_mean = (stats['mean'] * stats['count'] + global_mean * smooth) / (stats['count'] + smooth)
            train_encoded.iloc[val_idx] = val[feature].map(smooth_mean.to_dict()).fillna(global_mean)

        # test TE
        stats = train_df.groupby(feature)[target_col].agg(['mean', 'count'])
        smooth_mean = (stats['mean'] * stats['count'] + global_mean * smooth) / (stats['count'] + smooth)
        test_encoded[:] = test_df[feature].map(smooth_mean.to_dict()).fillna(global_mean)

        # save mean 
        train_df[f"{feature}_te"] = train_encoded
        test_df[f"{feature}_te"] = test_encoded

    print("Finish TE mean")
    
    return train_df, test_df

train, test = target_encode(train, test, te_cols, target_col="Heart Disease")

In [ ]:
object_cols = train.select_dtypes(include='object').columns.tolist()
train = train.drop(columns=object_cols,axis=1)
test = test.drop(columns=object_cols,axis=1)

# PCA Analysis

In [ ]:
pca_train = train.copy()
pca_test = test.copy()

In [ ]:
pca_train = pca_train.drop(columns="Heart Disease",axis=1)

pca_select_features = ["Age_scaled","BP_minmax","Cholesterol_scaled","Max HR_minmax","ST depression_robust"]

pca_train = pca_train[pca_select_features]
pca_test = pca_test[pca_select_features]

In [ ]:
from sklearn.decomposition import PCA
PCA_train = PCA()
pca_train_data = PCA_train.fit(pca_train)
pca_test_data = PCA_train.transform(pca_test)

In [ ]:
loadings = pd.DataFrame(PCA_train.components_, 
                        columns=pca_train.columns,  
                        index=[f'PCA{i+1}' for i in range(PCA_train.n_components_)])

In [ ]:
for i in range(PCA_train.n_components_):
    if i < 5:
        pc = loadings.iloc[i]
        print(f"Top features for PC{i+1}:")
        print(pc.abs().sort_values(ascending=False).head(3))  # top3
        print()

In [ ]:
explained_variance_ratio = PCA_train.explained_variance_ratio_
cumulative_explained_variance = explained_variance_ratio.cumsum()

In [ ]:
threshold = 0.95
n_components = np.argmax(cumulative_explained_variance >= threshold) + 1
print(n_components)

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(cumulative_explained_variance, marker='o', label='Cumulative Explained Variance')
plt.axhline(y=threshold, color='r', linestyle='--', label=f'{int(threshold*100)}% Threshold')
plt.axvline(x=n_components-1, color='g', linestyle='--', label=f'{n_components} Components')
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('PCA Cumulative Explained Variance')
plt.legend()
plt.grid(True)
plt.show()

print(f"Number of components to reach {threshold*100}% variance: {n_components}")

In [ ]:
num_selected_components = 3

train_pca = PCA(n_components=num_selected_components)

train_pca_result = train_pca.fit_transform(pca_train)
test_pca_result = train_pca.transform(pca_test)

In [ ]:
pca_columns = [f'PCA{i+1}' for i in range(num_selected_components)]
train_df_pca = pd.DataFrame(data=train_pca_result, columns=pca_columns)
test_df_pca = pd.DataFrame(data=test_pca_result, columns=pca_columns)

In [ ]:
train = pd.concat([train, train_df_pca], axis=1)
test = pd.concat([test, test_df_pca], axis=1)

In [ ]:
print("After PCA transformer train data shape is {}".format(train.shape))
print("After PCA transformer test data shape is {}".format(test.shape))

# K-Means

In [ ]:
from sklearn.cluster import MiniBatchKMeans
def auto_kmeans_features(train_df, test_df, feature_list, max_k=15):
    
    scaler = MinMaxScaler(feature_range=(-1, 1))
    
    train_scaled = scaler.fit_transform(train_df[feature_list])
    test_scaled = scaler.transform(test_df[feature_list])
    
    inertias = []
    k_range = range(1, max_k + 1)
    for k in k_range:
        km = MiniBatchKMeans(n_clusters=k, random_state=42, batch_size=10000, n_init=3)
        km.fit(train_scaled)
        inertias.append(km.inertia_)
    
    p1, p2 = np.array([k_range[0], inertias[0]]), np.array([k_range[-1], inertias[-1]])
    distances = [np.abs(np.cross(p2-p1, p1-np.array([k_range[i], inertias[i]]))) / np.linalg.norm(p2-p1) for i in range(len(k_range))]
    optimal_k = k_range[np.argmax(distances)]
    print(f" best K = {optimal_k}")

    
    plt.figure(figsize=(10, 6))
    plt.plot(k_range, inertias, 'bo-', label='Inertia (Within-cluster SSE)')
    plt.plot([k_range[0], k_range[-1]], [inertias[0], inertias[-1]], 'r--', label='Baseline (p1-p2)')
    plt.axvline(x=optimal_k, color='green', linestyle='--', label=f'Optimal K = {optimal_k}')
    plt.title('Elbow Method for Optimal K', fontsize=14)
    plt.xlabel('Number of Clusters (k)', fontsize=12)
    plt.ylabel('Inertia', fontsize=12)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()
    
    final_kmeans = MiniBatchKMeans(n_clusters=optimal_k, random_state=42, batch_size=10000)
    
    train_df['knn_cluster'] = final_kmeans.fit_predict(train_scaled)
    test_df['knn_cluster'] = final_kmeans.predict(test_scaled)
    
    
    train_dists = final_kmeans.transform(train_scaled)
    test_dists = final_kmeans.transform(test_scaled)
    
    for i in range(optimal_k):
        train_df[f'knn_dist_center_{i}'] = train_dists[:, i]
        test_df[f'knn_dist_center_{i}'] = test_dists[:, i]
        
    print(f"add {optimal_k + 1} features to train data")
    return train_df, test_df, final_kmeans


knn_features = nums_cols
train, test, model = auto_kmeans_features(train, test, knn_features)

# Weight of Evidence

In [ ]:
from category_encoders import WOEEncoder

def WoE(train_df,test_df,features,target_col):
    #
    train_woe = train_df.copy()
    test_woe = test_df.copy()
    #
    encode_list=features
    #
    encoder = WOEEncoder(cols=encode_list)
    encoder.fit(train_woe[encode_list],train_woe[target_col])
    #
    train_woe_part = encoder.transform(train_woe[encode_list]).add_suffix('_woe')
    test_woe_part = encoder.transform(test_woe[encode_list]).add_suffix('_woe')
    #
    train_final = pd.concat([train_df, train_woe_part], axis=1)
    test_final = pd.concat([test_df, test_woe_part], axis=1)
    
    return train_final, test_final

woe_features = ["Age_custom_bin","Sex","Chest pain type","BP_custom_bin","Cholesterol_custom_bin","FBS over 120","EKG results",
                "Max HR_custom_bin","Exercise angina","ST depression_custom_bin","Slope of ST","Number of vessels fluro","Thallium"]

train,test = WoE(train_df=train,test_df=test,features=woe_features,target_col='Heart Disease')

In [ ]:
def plot_woe_row(df, features):

    n_features = len(features)
    n_cols = 3
    n_rows = math.ceil(n_features / n_cols)
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 5 * n_rows))
    axes = axes.flatten() 
    

    sns.set_style("whitegrid")
    
    for i, col in enumerate(features):
        woe_col = f"{col}_woe"
        
        plot_data = df[[col, woe_col]].drop_duplicates().sort_values(by=woe_col)
        
        sns.barplot(x=col, y=woe_col, data=plot_data, ax=axes[i], palette="Spectral")
        
        axes[i].set_title(f'WoE: {col}', fontsize=14, fontweight='bold')
        axes[i].set_ylabel('Weight of Evidence')
        axes[i].axhline(0, color='black', linestyle='--', linewidth=1.2)
        axes[i].tick_params(axis='x', rotation=45) 
        
        for p in axes[i].patches:
            axes[i].annotate(f'{p.get_height():.2f}', 
                             (p.get_x() + p.get_width() / 2., p.get_height()), 
                             ha='center', va='bottom', fontsize=10)

    for j in range(i + 1, len(axes)):
        axes[j].axis('off')

    plt.tight_layout()
    plt.show()

plot_woe_row(train, features=woe_features)

In [ ]:
def create_woe_features(df,features):
    
    woe_cols = [f"{col}_woe" for col in features]
    df["Total_Clinical_Weight"] = df[woe_cols].sum(axis=1)
    # combine 
    df["is_Age_Cholesterol_custom_bin"] = (((df["Age_custom_bin"] >= 3)) & ((df["Cholesterol_custom_bin"].isin([3, 4])))).astype(int)
    df["is_EKG results_MaxHR"] = ((df["EKG results"] ==2) & (df["Max HR_custom_bin"]<3)).astype(int)
    df["is_ST depression_Slope_bin"] = ((df["ST depression_custom_bin"] >= 4) & (df["Slope of ST"] >=2)).astype(int)
    df["is_Thallium_Number of vessels fluro_bin"] = ((df["Thallium"] >= 6) & (df["Number of vessels fluro"] >=2)).astype(int)
    return df

train = create_woe_features(train,features=woe_features)
test = create_woe_features(test,features=woe_features)

In [ ]:
woe_features = ["Age_custom_bin_woe","Sex_woe","Chest pain type_woe","BP_custom_bin_woe","Cholesterol_custom_bin_woe",
                "FBS over 120_woe","EKG results_woe","Max HR_custom_bin_woe","Exercise angina_woe","ST depression_custom_bin_woe",
               "Slope of ST_woe","Number of vessels fluro_woe","Thallium_woe","Total_Clinical_Weight"]

plot_kde_by_label(train, nums_cols=woe_features, label_col='Heart Disease')

In [ ]:
train.head(2)

# Define Overlap Area 

In [ ]:
import optuna
def parameterized_correction(df, p):
    df_result = df.copy()
    is_ambiguous = df["Total_Clinical_Weight"].between(-2.5, 2.5)
    
    woe_components = (
        df["Age_custom_bin_woe"] * p['w_age'] +
        df["Sex_woe"] * p['w_sex'] +
        df["Chest pain type_woe"] * p['w_chest'] +
        df["BP_custom_bin_woe"] * p['w_bp'] +
        df["Cholesterol_custom_bin_woe"] * p['w_cho'] +
        df["FBS over 120_woe"] * p['w_fbs'] +
        df["EKG results_woe"] * p['w_ekg'] +
        df["Max HR_custom_bin_woe"] * p['w_max_hr'] +
        df["Exercise angina_woe"] * p['w_exercise'] +
        df["ST depression_custom_bin_woe"] * p['w_st'] +
        df["Slope of ST_woe"] * p['w_slope'] +
        df["Number of vessels fluro_woe"] * p['w_vessels'] +
        df["Thallium_woe"] * p['w_thal']
    )
    synergy_boost = np.sign(woe_components) * (woe_components ** 2) * p['w_non_linear']
    
    raw_correction = woe_components + synergy_boost
    final_correction = np.where(is_ambiguous, raw_correction, 0)
    
    if is_ambiguous.any():
        scaler = StandardScaler()
        mask = final_correction != 0
        if mask.any():
            # 這裡標準化後乘以 p['impact_scale']，讓 Optuna 控制修正幅度
            vals = final_correction[mask].reshape(-1, 1)
            final_correction[mask] = scaler.fit_transform(vals).flatten() * p['impact_scale']

    df_result["Ambiguous_Correction"] = final_correction
    return df_result

# Search Params

In [ ]:
from sklearn.linear_model import LogisticRegression

y = train["Heart Disease"]
def objective(trial):
    
    p = {
        'w_max_hr': trial.suggest_float('w_max_hr', 0.5, 5.0),
        'w_st': trial.suggest_float('w_st', 0.5, 5.0),
        'w_vessels': trial.suggest_float('w_vessels', 0.5, 5.0),
        'w_thal': trial.suggest_float('w_thal', 0.5, 5.0),
        
        'w_age': trial.suggest_float('w_age', 0.1, 3.0),
        'w_sex': trial.suggest_float('w_sex', 0.1, 3.0),
        'w_chest': trial.suggest_float('w_chest', 0.1, 3.0),
        'w_bp': trial.suggest_float('w_bp', 0.1, 3.0),
        'w_cho': trial.suggest_float('w_cho', 0.1, 3.0),
        'w_fbs': trial.suggest_float('w_fbs', 0.1, 3.0),
        'w_ekg': trial.suggest_float('w_ekg', 0.1, 3.0),
        'w_exercise': trial.suggest_float('w_exercise', 0.1, 3.0),
        'w_slope': trial.suggest_float('w_slope', 0.1, 3.0),
        'w_non_linear':trial.suggest_float('w_non_linear', 0.001, 0.1, log=True),
        
        'impact_scale': trial.suggest_float('impact_scale', 0.1, 3),
        
        'lower_bound': trial.suggest_float('lower_bound', -2.5, -1.0),
        'upper_bound': trial.suggest_float('upper_bound', 1.0, 2.5)
    }

    df_train_temp = parameterized_correction(train, p)
    
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    aucs = []
    
    features = ['Total_Clinical_Weight', 'Ambiguous_Correction']
    
    for train_idx, val_idx in skf.split(df_train_temp, y):
        X_tr, X_val = df_train_temp.iloc[train_idx][features], df_train_temp.iloc[val_idx][features]
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
        
        model = LogisticRegression(C=1.0).fit(X_tr, y_tr)
        probs = model.predict_proba(X_val)[:, 1]
        aucs.append(roc_auc_score(y_val, probs))
    
    return np.mean(aucs)

In [ ]:
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)

print("Best AUC:", study.best_value)
print("Best Set is :", study.best_params)

best_p = study.best_params
train = parameterized_correction(train, best_p)
test = parameterized_correction(test, best_p)

In [ ]:
train.head(2)

In [ ]:
def correction_features(df):
    df['in_overlap_zone'] = df['Total_Clinical_Weight'].between(-5, 5).astype(np.int8)
    df['correction_intensity'] = df['Ambiguous_Correction'] - df['Total_Clinical_Weight']
    df['is_upward_correction'] = (df['Ambiguous_Correction'] > df['Total_Clinical_Weight']).astype(np.int8)
    return df

train = correction_features(train)
test = correction_features(test)

In [ ]:
from sklearn.model_selection import StratifiedKFold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

auc_orig = []
auc_new = []
features_orig = ['Total_Clinical_Weight']
features_new = ['Total_Clinical_Weight','Ambiguous_Correction','in_overlap_zone','correction_intensity','is_upward_correction']

for train_idx, val_idx in skf.split(train, y):
    
    X_train, X_val = train.iloc[train_idx], train.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    m1 = LogisticRegression().fit(X_train[features_orig], y_train)
    auc_orig.append(roc_auc_score(y_val, m1.predict_proba(X_val[features_orig])[:, 1]))
    
    m2 = LogisticRegression().fit(X_train[features_new], y_train)
    auc_new.append(roc_auc_score(y_val, m2.predict_proba(X_val[features_new])[:, 1]))

print(f"Only Total_Clinical_Weight AUC score is : {np.mean(auc_orig):.5f}")
print(f"Add Ambiguous_Correction AUC score is : {np.mean(auc_new):.5f}")

In [ ]:
importance = m2.coef_[0]
feature_names = ['Total_Clinical_Weight','Ambiguous_Correction','in_overlap_zone','correction_intensity','is_upward_correction']

weight_df = pd.DataFrame({
    'Feature': feature_names,
    'Weight (Coefficient)': importance,
    'Absolute Weight': np.abs(importance)
}).sort_values(by='Absolute Weight', ascending=False)

print(weight_df)

plt.figure(figsize=(8, 5))
sns.barplot(x='Weight (Coefficient)', y='Feature', data=weight_df, palette='viridis')
plt.title('Final Model Feature Weights (Beta Coefficients)')
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.show()

In [ ]:
intercept = m2.intercept_[0]
coef_total = m2.coef_[0][0]
coef_corr1 = m2.coef_[0][1]
coef_corr2 = m2.coef_[0][2]
coef_corr3 = m2.coef_[0][3]
coef_corr4 = m2.coef_[0][4]



print(f"Intercept : {intercept:.4f}")
print(f"Total_Clinical_Weight (w1) : {coef_total:.4f}")
print(f"Ambiguous_Correction (w2) : {coef_corr1:.4f}")
print(f"is_upward_correction (w2) : {coef_corr2:.4f}")
print(f"in_overlap_zone (w2) : {coef_corr3:.4f}")
print(f"correction_intensity (w2) : {coef_corr4:.4f}")



print(f"\nformula(z_score):")
print(f"z = {intercept:.4f} + ({coef_total:.4f} * Total) + ({coef_corr1:.4f} * Correction) + ({coef_corr2:.4f} * Correction) + ({coef_corr3:.4f} * Correction) + ({coef_corr4:.4f} * Correction)")
print("P = 1 / (1 + exp(-z))")

In [ ]:
x_vals = np.array([-5, 5])
y_vals = -(coef_total / coef_corr) * x_vals - (intercept / coef_corr)

plt.figure(figsize=(8, 6))
plt.plot(x_vals, y_vals, '--r', label='Decision Boundary (P=0.5)')
plt.scatter(train['Total_Clinical_Weight'], train['Ambiguous_Correction'], 
            c=y, cmap='coolwarm', alpha=0.5)
plt.xlabel('Total_Clinical_Weight')
plt.ylabel('Ambiguous_Correction')
plt.title('How the two features split the data')
plt.legend()
plt.show()

In [ ]:
def add_decimal_bin_features(df, cols, bins=10):
    
    df = df.copy()
    for col in cols:
        if df[col].nunique(dropna=True) <= 1:
            print(f"[Skip] {col} constant or all NaN, skip feature engineering")
            continue
        
        x = df[col]

        df[f"{col}_frac"] = x % 1
        
        df[f"{col}_is_int"] = ((x % 1) == 0).astype(np.int8)
        
        df[f"{col}_r1"] = x.round(1)
        df[f"{col}_r2"] = x.round(2)
        df[f"{col}_r3"] = x.round(3)
        
        df[f"{col}_int"] = x.astype(int)

        df[f"{col}_int_frac"] = df[f"{col}_int"] * df[f"{col}_frac"]
        
    
    return df

cols = ["Total_Clinical_Weight","Ambiguous_Correction","correction_intensity"]
train = add_decimal_bin_features(train, cols)
test  = add_decimal_bin_features(test,  cols)

# Features Selection

In [ ]:
Y = train["Heart Disease"]
select_train = train.drop(columns="Heart Disease",axis=1)

In [ ]:
select_train.shape

In [ ]:
import xgboost as xgb
from operator import itemgetter
from xgboost import plot_importance

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

xgb_params = {
        'n_jobs': -1,
        'objective':'binary:logistic',
        'tree_method': 'hist',       # GPU
        'device': 'cuda',
        'verbosity': 0,
        'random_state': 42,
        'max_depth':6,
        'learning_rate':0.01,
        'subsample':0.8,
        'reg_lambda':3,
        'colsample_bytree':0.8,
        'n_estimators':500,
        'min_child_weight':5
    }

xgb_model = xgb.XGBClassifier(**xgb_params)
xgb_model.fit(select_train,Y)

importance_scores = xgb_model.feature_importances_
importance_gains = xgb_model.get_booster().get_score(importance_type='gain')
ranked_features = pd.Series(importance_gains, index=select_train.columns).sort_values(ascending=False).index.tolist()

plot_importance(xgb_model, importance_type='gain', max_num_features=10)
plt.show()

print("Total Data {}".format(select_train.shape))
print("Features Select by gain is {}".format(len(importance_gains)))

In [ ]:
from sklearn.model_selection import cross_val_score
from xgboost import XGBClassifier
import xgboost as xgb

def evaluate_feature_subsets(X, y, ranked_features, model, steps, cv_strategy, plot_limit=len(importance_gains)):
    results = {}
    total_available = len(ranked_features)
    print(f"Total available features: {total_available}")
    
    for k in steps:
        actual_k = min(k, total_available)
        feats = ranked_features[:actual_k]
        
        
        scores = cross_val_score(model, X[feats], y, cv=cv_strategy, scoring='roc_auc', n_jobs=-1)
        avg_auc = np.mean(scores)
        results[actual_k] = avg_auc
        
        print(f"Top {actual_k:3d} features -> ROC-AUC: {avg_auc:.5f}")
        
        if actual_k == total_available:
            break

    plt.figure(figsize=(8, 5))
    x_values = list(results.keys())
    y_values = list(results.values())
    
    sns.lineplot(x=x_values, y=y_values, marker='o', color='#2c3e50', linewidth=2)
    
    if plot_limit in x_values or (min(x_values) < plot_limit < max(x_values)):
        plt.axvline(x=plot_limit, color='#e74c3c', linestyle='--', label=f'Original Limit ({plot_limit})')
    
    best_k = max(results, key=results.get)
    best_auc = results[best_k]
    
    plt.annotate(
        f'Best: {best_k}\nAUC: {best_auc:.4f}', 
        xy=(best_k, best_auc), 
        xytext=(15, -15),             
        textcoords='offset points',   
        arrowprops=dict(
            arrowstyle='->',          
            connectionstyle='arc3',  
            color='black'
        ),
        fontsize=10,
        fontweight='bold'
    )

    plt.title('Feature Selection: ROC-AUC Trend', fontsize=12)
    plt.xlabel('Features Count', fontsize=10)
    plt.ylabel('Mean ROC-AUC', fontsize=10)
    plt.legend(prop={'size': 9})
    plt.grid(True, linestyle=':', alpha=0.5)
    plt.tight_layout()
    plt.show()
    
    return results

steps = [10,20,25, 30, 35, 40, 45, 50, 55, 60, 65, 75, 85, 95, 100, 105,110,115,120,125,130,135,140,145,150,155,160,165,169]
auc_results = evaluate_feature_subsets(select_train, Y, ranked_features, xgb_model, steps, skf)

In [ ]:
train.head(2)

In [ ]:
final_k=75

final_select_features = ranked_features[:final_k]
linear_select_features = ['Total_Clinical_Weight','Ambiguous_Correction','in_overlap_zone','correction_intensity','is_upward_correction']

X_linear = train[linear_select_features]
test_linear = test[linear_select_features]

X = select_train[final_select_features]
test = test[final_select_features]


print("We choose {} features for trees".format(len(final_select_features)))
print("We choose {} features for linear".format(len(linear_select_features)))

print("The train data shape is {}".format(X.shape))
print("The test data shape is {}".format(test.shape))
print("The train_linear data shape is {}".format(X_linear.shape))
print("The test_linear data shape is {}".format(test_linear.shape))
print("The Target shape is {}".format(Y.shape))

# Hyperparameter Search

In [ ]:
import optuna
# trees_number     -> how many tree
# tree_complexity  -> max_depth
# regularization -> overfitting control
# row_sampling -> data random
# col_sampling -> features random
# learning_rate -> step

def suggest_shared_params(trial):
    return {
        # scale
        "trees_number":trial.suggest_int("trees_number",100,3000),
        "learning_rate":trial.suggest_float("learning_rate",0.005,0.2,log=True),
        # structure
        "tree_complexity":trial.suggest_int("tree_complexity",3,12),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 100), 
        # re
        "regularization":trial.suggest_float("regularization",1e-3,10.0,log=True),
        # random
        "row_sampling":trial.suggest_float("row_sampling",0.5,1.0),
        "col_sampling":trial.suggest_float("col_sampling",0.5,1.0)
    
    }

# CV Evaluation Function

In [ ]:
from sklearn.metrics import roc_auc_score

def cv_roc_auc(model_builder, X, Y, kf):
    
    oof_probs = np.zeros(len(X))

    for tr_idx, val_idx in kf.split(X, Y):
        model = model_builder()
        model.fit(X.iloc[tr_idx], Y.iloc[tr_idx])
        
        if hasattr(model, "predict_proba"):
            oof_probs[val_idx] = model.predict_proba(X.iloc[val_idx])[:, 1]
        else:
            oof_probs[val_idx] = model.predict(X.iloc[val_idx])

    auc_score = roc_auc_score(Y, oof_probs)
    return auc_score

# Define Search Model

In [ ]:
from xgboost import XGBClassifier

def build_xgb_clf(shared):
    max_leaves_val = 2 ** int(shared["tree_complexity"])
    
    return XGBClassifier(
        objective="binary:logistic",
        eval_metric="auc",
        tree_method="hist",
        grow_policy="lossguide",
        device="cuda",
        random_state=42,
        n_jobs=-1,

        n_estimators=int(shared["trees_number"]),
        learning_rate=shared["learning_rate"],

        max_leaves=max_leaves_val,
        min_child_weight=shared["min_child_samples"], 

        reg_lambda=shared["regularization"],
        
        subsample=shared["row_sampling"],
        colsample_bytree=shared["col_sampling"]
    )

# Optuna Objective

In [ ]:
def objective(trial):
    shared = suggest_shared_params(trial)
    
    from sklearn.model_selection import StratifiedKFold
    
    kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    auc_score = cv_roc_auc(lambda: build_xgb_clf(shared), X, Y, kf)

    return auc_score

# Start Optuna

In [ ]:
study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(
        multivariate=True,
        group=True,
        seed=42
    )
)

#study.optimize(objective, n_trials=50)

In [ ]:
best_params = {'trees_number': 2363,
               'learning_rate': 0.024925996637526548,
               'tree_complexity': 3, 
               'min_child_samples': 17, 
               'regularization': 0.006704977748804664,
               'row_sampling': 0.8948981331774478, 
               'col_sampling': 0.983089681473579}

In [ ]:
#optuna.visualization.plot_optimization_history(study).show()
#optuna.visualization.plot_parallel_coordinate(study, params=['learning_rate','trees_number', 'tree_complexity', 'regularization','row_sampling','col_sampling']).show()

# Train Model

In [ ]:
from sklearn.metrics import roc_auc_score
import numpy as np

def hill_climbing_auc(x_oof, y, x_test):
    scores = {}
    for col in x_oof.columns:
        scores[col] = roc_auc_score(y, x_oof[col])

    scores = {k: v for k, v in sorted(scores.items(), key=lambda item: item[1], reverse=True)}

    x_oof = x_oof[list(scores.keys())]
    x_test = x_test[list(scores.keys())]

    STOP = False
    current_best_ensemble = x_oof.iloc[:, 0].copy()
    current_best_test_preds = x_test.iloc[:, 0].copy()
    
    final_weights = {list(scores.keys())[0]: 1.0}
    for col in list(scores.keys())[1:]:
        final_weights[col] = 0.0

    MODELS = x_oof.iloc[:, 1:].copy()
    weight_range = np.arange(0, 0.51, 0.01) 
    history = [roc_auc_score(y, current_best_ensemble)]
    
    while not STOP:
        potential_new_best_cv_score = roc_auc_score(y, current_best_ensemble)
        k_best, wgt_best = None, None
        
        for k in MODELS.columns:
            for wgt in weight_range:
                potential_ensemble = (1 - wgt) * current_best_ensemble + wgt * MODELS[k]
                cv_score = roc_auc_score(y, potential_ensemble)
                
                if cv_score > potential_new_best_cv_score: 
                    potential_new_best_cv_score = cv_score
                    k_best, wgt_best = k, wgt

        if k_best is not None:
            for model_name in final_weights:
                final_weights[model_name] *= (1 - wgt_best)
            final_weights[k_best] += wgt_best
            
            current_best_ensemble = (1 - wgt_best) * current_best_ensemble + wgt_best * MODELS[k_best]
            current_best_test_preds = (1 - wgt_best) * current_best_test_preds + wgt_best * x_test[k_best]
            
            MODELS.drop(k_best, axis=1, inplace=True)
            if MODELS.shape[1] == 0:
                STOP = True
            history.append(potential_new_best_cv_score)
        else:
            STOP = True

    return {
        "oof_pred": current_best_ensemble,
        "test_pred": current_best_test_preds,
        "weights": final_weights,
        "final_auc": history[-1]
    }

In [ ]:
import warnings
warnings.filterwarnings("ignore")
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression, RidgeClassifier,SGDClassifier,PassiveAggressiveClassifier, Perceptron
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.naive_bayes import GaussianNB
from scipy.special import expit
import time

class EnsembleModelTrainer:
    def __init__(self,shared_best_params,n_splits=5,random_state=42):
        self.shared = shared_best_params
        self.n_splits = n_splits
        self.random_state = random_state
        self.skf = StratifiedKFold(n_splits=self.n_splits,shuffle=True,random_state=self.random_state)

        self.scores = {
            "lr":[],"lrl2":[],
            "ridge":[],"lsvc":[],"sgd":[],"pa":[],"lda":[],"gnb":[],
            "xgb_base":[],"xgb_grow_policy":[],"xgb_subspace":[],"xgb_re":[],"xgb_inter":[],"xgb_lrpath":[],
            "lgb_base":[],"lgb_subspace":[],"lgb_re":[],"lgb_lrpath":[],
            "hgb_base":[],"hgb_lr":[],"hgb_trees":[],"hgb_depth":[],
            "dt":[],"cat_base":[],"cat_lr":[],"cat_deep":[],
            "ens":[],"hill":[],"stack":[]
        }

        self.test_preds = {
            "lr":[],"lrl2":[],
            "ridge":[],"lsvc":[],"sgd":[],"pa":[],"lda":[],"gnb":[],
            "xgb_base":[],"xgb_grow_policy":[],"xgb_subspace":[],"xgb_re":[],"xgb_inter":[],"xgb_lrpath":[],
            "lgb_base":[],"lgb_subspace":[],"lgb_re":[],"lgb_lrpath":[],
            "hgb_base":[],"hgb_lr":[],"hgb_trees":[],"hgb_depth":[],
            "dt":[],"cat_base":[],"cat_lr":[],"cat_deep":[],
            "ens":[],"hill":[],"stack":[]
        }

        self.hill_weights=[]
    # predict------------------------------
    def _predict_proba_safe(self,model,X):
        if hasattr(model,"predict_proba"):
            return model.predict_proba(X)[:,1]
        elif hasattr(model,"decision_function"):
            return expit(model.decision_function(X))
        else:
            raise ValueError("Model has no probability output")

    # model------------------------
    def _build_linear_models(self):
        return {
            "lr":Pipeline([
                ("scaler",StandardScaler()),
                ("lr",LogisticRegression(
                    C=1.0,
                    random_state=self.random_state
                ))
            ]),
            "lrl2":Pipeline([
                ("scaler",StandardScaler()),
                ("lrl1",LogisticRegression(
                    penalty="l2",
                    C=0.1,
                    random_state=self.random_state
                ))
            ]),
            "ridge":Pipeline([
                ("scaler",StandardScaler()),
                ("ridge",RidgeClassifier(
                    alpha=1.0
                ))
            ]),
            "lsvc": Pipeline([
                ("scaler", StandardScaler()),
                ("lsvc", CalibratedClassifierCV(LinearSVC(dual=False, C=0.5, random_state=self.random_state)))
            ]),
            "sgd": Pipeline([
                ("scaler", StandardScaler()),
                ("sgd", SGDClassifier(loss='modified_huber', penalty='elasticnet', random_state=self.random_state))
            ]),
            "pa": Pipeline([
                ("scaler", StandardScaler()),
                ("pa", PassiveAggressiveClassifier(max_iter=1000, random_state=self.random_state))
            ]),
            "lda": Pipeline([
                ("scaler", StandardScaler()),
                ("lda", LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto"))
            ]),
            "gnb": Pipeline([
                ("scaler", StandardScaler()),
                ("gnb", GaussianNB(var_smoothing=1e-8))
            ]),

        }
    def _build_tree_models(self):
        s = self.shared
        return {
            "xgb_base": XGBClassifier(
                tree_method="hist",
                objective="binary:logistic",
                device="cuda",
                learning_rate=s["learning_rate"],
                n_estimators=s["trees_number"],
                max_depth=s["tree_complexity"],
                subsample=s["row_sampling"],
                colsample_bytree=s["col_sampling"],
                reg_lambda=s["regularization"],
                min_child_weight=s["min_child_samples"],
                eval_metric="auc",
                random_state=42,
                n_jobs=-1
            ),
            "xgb_grow_policy": XGBClassifier(
                tree_method="hist",
                grow_policy="lossguide",
                max_leaves=2 ** (s["tree_complexity"] + 1),
                objective="binary:logistic",
                device="cuda",
                learning_rate=s["learning_rate"],
                n_estimators=s["trees_number"],
                subsample=s["row_sampling"],
                colsample_bytree=s["col_sampling"],
                reg_lambda=s["regularization"],
                n_jobs=-1,
                random_state=142
            ),
            "xgb_subspace": XGBClassifier(
                tree_method="hist",
                objective="binary:logistic",
                device="cuda",
                learning_rate=s["learning_rate"],
                n_estimators=s["trees_number"] + 3,
                max_depth=s["tree_complexity"],
                subsample=s["row_sampling"],
                colsample_bytree=s["col_sampling"] - 0.15,
                reg_lambda=s["regularization"],
                min_child_weight=s["min_child_samples"],
                eval_metric="auc",
                random_state=242,
                n_jobs=-1
            ),
            "xgb_re": XGBClassifier(
                tree_method="hist",
                objective="binary:logistic",
                device="cuda",
                learning_rate=s["learning_rate"],
                n_estimators=s["trees_number"],
                max_depth=s["tree_complexity"] + 3,
                subsample=s["row_sampling"],
                colsample_bytree=s["col_sampling"],
                reg_lambda=s["regularization"] * 3,
                min_child_weight=s["min_child_samples"],
                eval_metric="auc",
                random_state=self.random_state,
                n_jobs=-1
            ),
            "xgb_inter": XGBClassifier(
                tree_method="hist",
                objective="binary:logistic",
                device="cuda",
                learning_rate=s["learning_rate"],
                n_estimators=s["trees_number"],
                max_depth=s["tree_complexity"] + 2,
                subsample=s["row_sampling"],
                colsample_bytree=s["col_sampling"],
                reg_lambda=s["regularization"],
                min_child_weight=s["min_child_samples"] * 0.7,
                eval_metric="auc",
                random_state=self.random_state,
                n_jobs=-1
            ),
            "xgb_lrpath": XGBClassifier(
                tree_method="hist",
                objective="binary:logistic",
                device="cuda",
                learning_rate=s["learning_rate"] * 0.7,
                n_estimators=int(s["trees_number"] * 1.4),
                max_depth=s["tree_complexity"],
                subsample=s["row_sampling"],
                colsample_bytree=s["col_sampling"],
                reg_lambda=s["regularization"],
                min_child_weight=s["min_child_samples"],
                eval_metric="auc",
                random_state=self.random_state,
                n_jobs=-1
            ),
            "lgb_base": LGBMClassifier(
                objective="binary",
                device='gpu',
                learning_rate=s["learning_rate"],
                n_estimators=s["trees_number"],
                max_depth=s["tree_complexity"],
                subsample=s["row_sampling"],
                colsample_bytree=s["col_sampling"],
                lambda_l2=s["regularization"],
                random_state=52,
                n_jobs=-1,
                verbose=-1,
                verbosity=-1
            ),
            "lgb_subspace": LGBMClassifier(
                objective="binary",
                device='gpu',
                learning_rate=s["learning_rate"],
                n_estimators=s["trees_number"],
                max_depth=s["tree_complexity"],
                subsample=s["row_sampling"] - 0.15,
                colsample_bytree=s["col_sampling"],
                lambda_l2=s["regularization"],
                random_state=152,
                n_jobs=-1,
                verbose=-1,
                verbosity=-1
            ),
            "lgb_re": LGBMClassifier(
                objective="binary",
                device='gpu',
                learning_rate=s["learning_rate"],
                n_estimators=s["trees_number"],
                max_depth=s["tree_complexity"],
                subsample=s["row_sampling"],
                colsample_bytree=s["col_sampling"],
                lambda_l2=s["regularization"] * 3,
                random_state=252,
                n_jobs=-1,
                verbose=-1,
                verbosity=-1
            ),
            "lgb_lrpath": LGBMClassifier(
                objective="binary",
                device='gpu',
                learning_rate=s["learning_rate"] * 0.7,
                n_estimators=int(s["trees_number"] * 1.4),
                max_depth=s["tree_complexity"],
                subsample=s["row_sampling"],
                colsample_bytree=s["col_sampling"],
                lambda_l2=s["regularization"],
                random_state=352,
                n_jobs=-1,
                verbose=-1,
                verbosity=-1
            ),
            "cat_base": CatBoostClassifier(
                iterations=s["trees_number"],
                learning_rate=s["learning_rate"],
                depth=min(s["tree_complexity"], 8),
                l2_leaf_reg=s["regularization"],
                loss_function="Logloss",
                eval_metric="AUC",
                verbose=False,
                task_type="GPU",
                random_seed=62
            ),
            "cat_lr": CatBoostClassifier(
                iterations=s["trees_number"],
                learning_rate=s["learning_rate"] * 0.5,
                depth=8,
                l2_leaf_reg=s["regularization"],
                loss_function="Logloss",
                eval_metric="AUC",
                verbose=False,
                task_type="GPU",
                random_seed=162
            ),
            "cat_deep": CatBoostClassifier(
                iterations=int(s["trees_number"]*1.2),
                learning_rate=s["learning_rate"],
                depth=s["tree_complexity"]*3,
                l2_leaf_reg=s["regularization"]*3,
                loss_function="Logloss",
                eval_metric="AUC",
                verbose=False,
                task_type="GPU",
                random_seed=262
            ),
            "hgb_base": HistGradientBoostingClassifier(
                learning_rate=s["learning_rate"],
                max_iter=s["trees_number"],
                max_depth=s["tree_complexity"],
                l2_regularization=s["regularization"],
                random_state=72
            ),
            "hgb_lr": HistGradientBoostingClassifier(
                learning_rate=s["learning_rate"] * 0.5,
                max_iter=s["trees_number"],
                max_depth=s["tree_complexity"],
                l2_regularization=s["regularization"],
                random_state=172
            ),
            "hgb_trees": HistGradientBoostingClassifier(
                learning_rate=s["learning_rate"],
                max_iter=int(s["trees_number"] * 1.2),
                max_depth=s["tree_complexity"],
                l2_regularization=s["regularization"],
                random_state=272
            ),
            "hgb_depth": HistGradientBoostingClassifier(
                learning_rate=s["learning_rate"],
                max_iter=s["trees_number"] ,
                max_depth=s["tree_complexity"] * 2,
                l2_regularization=s["regularization"],
                random_state=372
            ),
            "dt": DecisionTreeClassifier(
            max_depth=s["tree_complexity"],
            min_samples_leaf=max(1, int(1 / max(s["row_sampling"], 1e-3))),
            ccp_alpha=s["regularization"] * 1e-2,
            random_state=82
            )   
        }
    def _get_models(self):
        models = {}
        models.update(self._build_linear_models())
        models.update(self._build_tree_models())
        return models
        
    # Training-------------------------------------
    def train_and_evaluate(self, X_linear,X_test_linear,X_tree,X_test_tree,Y, hill_climbing_func):
        print(f'{"-"*30} Training Started {"-"*30}')

        for fold, (tr_idx, va_idx) in enumerate(self.skf.split(Y, Y), 1):
            print(f'\n{"-"*20} Fold {fold} {"-"*20}')

            X_tr_lin, X_va_lin = X_linear.iloc[tr_idx], X_linear.iloc[va_idx]
            X_tr_tree, X_va_tree = X_tree.iloc[tr_idx], X_tree.iloc[va_idx]
            y_tr, y_va = Y.iloc[tr_idx], Y.iloc[va_idx]

            models = self._get_models()
            fold_val_preds = {}
            fold_test_preds = {}

            # ----- Base models -----
            for name, model in models.items():
                start = time.time()
                # 根據模型類型選特徵
                if name in self._build_linear_models().keys():
                    model.fit(X_tr_lin, y_tr)
                    val_pred = self._predict_proba_safe(model, X_va_lin)
                    test_pred = self._predict_proba_safe(model, X_test_linear)
                else:
                    model.fit(X_tr_tree, y_tr)
                    val_pred = self._predict_proba_safe(model, X_va_tree)
                    test_pred = self._predict_proba_safe(model, X_test_tree)

                
                score = roc_auc_score(y_va, val_pred)
                self.scores[name].append(score)
                fold_val_preds[name] = val_pred
                fold_test_preds[name] = test_pred

                print(f"{name.upper():5s} ROC AUC: {score:.5f} | time {time.time()-start:.1f}s")

            # ----- Simple Average -----
            ens_val = np.mean(list(fold_val_preds.values()), axis=0)
            ens_score = roc_auc_score(y_va, ens_val)
            self.scores["ens"].append(ens_score)

            ens_test = np.mean(list(fold_test_preds.values()), axis=0)
            self.test_preds["ens"].append(ens_test)

            print(f"ENS   ROC AUC: {ens_score:.5f}")

            # ----- Hill Climbing -----
            df_val = pd.DataFrame(fold_val_preds)
            df_test = pd.DataFrame(fold_test_preds)

            res = hill_climbing_func(df_val, y_va, df_test)
            hill_val = res["oof_pred"]
            hill_test = res["test_pred"]
            hill_weights = res["weights"]
            self.hill_weights.append(hill_weights)
            hill_score = roc_auc_score(y_va, hill_val)

            self.scores["hill"].append(hill_score)
            self.test_preds["hill"].append(hill_test)

            print(f"HILL  ROC AUC: {hill_score:.5f}")

            # ----- Stacking -----
            df_val = pd.DataFrame(fold_val_preds)
            df_test = pd.DataFrame(fold_test_preds)
            
            # Rank transform
            df_val_rank = df_val.rank(pct=True)
            df_test_rank = df_test.rank(pct=True)
            
            meta = LogisticRegression()
            meta.fit(df_val_rank.values, y_va)

            stack_val = meta.predict_proba(df_val_rank.values)[:, 1]
            stack_score = roc_auc_score(y_va, stack_val)

            self.scores["stack"].append(stack_score)

            stack_test = meta.predict_proba(df_test_rank.values)[:, 1]
            self.test_preds["stack"].append(stack_test)

            print(f"STACK ROC AUC: {stack_score:.5f}")

        self._print_summary()

    # ---------- Summary ----------
    def _print_summary(self, plot=True):
        print(f'\n{"-"*30} Final CV Results {"-"*30}')

        model_names = []
        mean_scores = []
        std_scores = []

        for k, v in self.scores.items():
            if v:
                mean_score = np.mean(v)
                std_score = np.std(v)

                model_names.append(k.upper())
                mean_scores.append(mean_score)
                std_scores.append(std_score)

                print(f"{k.upper():5s} ROC AUC: {mean_score:.5f} (+/- {std_score:.5f})")

        if plot and model_names:
            plt.figure(figsize=(12, 4))
            plt.plot(model_names, mean_scores, marker="o")
            plt.xlabel("Model")
            plt.ylabel("Mean ROC AUC")
            plt.title("CV Performance by Model")
            plt.xticks(rotation=45, ha="right")
            plt.grid(True)
            plt.show()

    # ---------- Final Prediction ----------
    def get_final_predictions(self, method="hill"):
        return np.mean(self.test_preds[method], axis=0)

    #---------- Final hill climbing weights ----------
    def get_final_hill_weights(self):

        all_models = self.hill_weights[0].keys()
        avg_weights = {}
        for m in all_models:
            avg_weights[m] = np.mean([fw[m] for fw in self.hill_weights])
        return avg_weights

    #---------- Get hill climbing weights ----------
    def get_final_hill_weights(self):
        df = pd.DataFrame(self.hill_weights)
        df.loc['mean'] = df.mean()
        return df

In [ ]:
trainer = EnsembleModelTrainer(
    shared_best_params=best_params,
    n_splits=5
)

trainer.train_and_evaluate(
    X_linear=X_linear,
    X_tree = X,
    X_test_linear=test_linear,
    X_test_tree=test,
    Y=Y,
    hill_climbing_func=hill_climbing_auc
)

hill_pred  = trainer.get_final_predictions("hill")
stack_pred = trainer.get_final_predictions("stack")
ens_pred   = trainer.get_final_predictions("ens")

# Model Analysis

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_linear)

lr_model = LogisticRegression()
lr_model.fit(X_scaled, Y)

intercept = lr_model.intercept_[0]
coefs = lr_model.coef_[0]
odds_ratios = np.exp(coefs)

lr_features = X_linear.columns.tolist() 

interpretation_df = pd.DataFrame({
    'Feature': lr_features,
    'Coefficient (w)': coefs,
    'Odds Ratio (exp(w))': odds_ratios
})

print(interpretation_df)

In [ ]:
#plt.figure(figsize=(8, 5))
#sns.barplot(x='Coefficient (w)', y='Feature', data=interpretation_df, palette='viridis')
#plt.title('Feature Importance (Standardized Coefficients)')
#plt.axvline(0, color='black', linestyle='--')
#plt.show()

In [ ]:
import xgboost as xgb
xgb_model = xgb.XGBClassifier(
            tree_method='hist',
            grow_policy='lossguide',     
            objective='binary:logistic',
            device='cuda',
            booster='gbtree',
            learning_rate=0.024925996637526548,
            n_estimators=2363,
            max_depth=0,                 
            max_leaves=256,             
            colsample_bytree=0.8,
            eval_metric='logloss',
            subsample=0.8,
            random_state=42,
            n_jobs=-1)
xgb_model.fit(X,Y)

In [ ]:
from operator import itemgetter
from xgboost import plot_importance
from xgboost import plot_importance

importance_dict = xgb_model.get_booster().get_score(importance_type='gain')

importance_list = sorted(importance_dict.items(), key=itemgetter(1), reverse=True)

plot_importance(xgb_model, importance_type='gain', max_num_features=10)
plt.show()

# Submission

In [ ]:
submission["Heart Disease"] = hill_pred
submission.head()

In [ ]:
plt.hist(submission["Heart Disease"],bins=100)
plt.title('Test Preds')
plt.ylim((0,10_000))
plt.show()

In [ ]:
submission.to_csv("hill_pred13.csv",index=False)